# RotoWire Bundesliga Predicted Lineups

This notebook opens the RotoWire Bundesliga lineups page through one undetected Chrome session, extracts all nine match cards, and saves the selected XI for both teams in each match.

Only players between the lineup-status row and the Injuries heading are collected. Injury-list players are excluded, while a QUES or OUT flag displayed on a selected starter is retained. Predicted and subsequently confirmed lineups are both accepted and labeled with RotoWire's displayed status. If RotoWire lists fewer than 11 selected starters, the missing slots are retained as question-mark placeholder players.

The validated UTF-8 JSON snapshot is written to outputs/rotowire/predicted_lineups through project_paths.py.

## 1. Resolve the project and output location

Project discovery supports JupyterLab and VS Code without embedding a machine-specific path.

In [1]:
# Import the libraries required by this notebook step.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import ROTOWIRE_PREDICTED_LINEUPS_DIR, ensure_directory

output_directory = ensure_directory(ROTOWIRE_PREDICTED_LINEUPS_DIR)
print(f"RotoWire lineup output directory: {output_directory}")

RotoWire lineup output directory: C:\kickbase project\outputs\rotowire\predicted_lineups


## 2. Imports and configuration

Chrome 150 matches the other scraping notebooks in this project. The notebook does not use requests.

In [2]:
# Import the libraries required by this notebook step.
import json
import re
from datetime import datetime
from typing import Any
from urllib.parse import urljoin

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing from this notebook kernel. Install them with: "
        "%pip install beautifulsoup4 undetected-chromedriver selenium"
    ) from exc

# Set workflow configuration value: SOURCE_NAME.
SOURCE_NAME = "RotoWire"
# Set workflow configuration value: SOURCE_URL.
SOURCE_URL = "https://www.rotowire.com/soccer/lineups.php?league=BUND"
# Set workflow configuration value: LEAGUE_CODE.
LEAGUE_CODE = "BUND"
# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 45
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 30
# Set workflow configuration value: EXPECTED_MATCH_COUNT.
EXPECTED_MATCH_COUNT = 9
# Set workflow configuration value: EXPECTED_TEAM_COUNT.
EXPECTED_TEAM_COUNT = 18
# Set workflow configuration value: EXPECTED_STARTERS_PER_TEAM.
EXPECTED_STARTERS_PER_TEAM = 11
# Set workflow configuration value: EXPECTED_PLAYER_COUNT.
EXPECTED_PLAYER_COUNT = 198
# Set workflow configuration value: LINEUP_CARD_SELECTOR.
LINEUP_CARD_SELECTOR = "div.lineup.is-soccer"
# Set workflow configuration value: ALLOWED_STARTER_INJURY_STATUSES.
ALLOWED_STARTER_INJURY_STATUSES = {None, "QUES", "OUT"}

## 3. Parsing helpers

The parser uses direct list children to stop at the Injuries heading instead of assuming that every lineup list contains only 11 player elements.

In [3]:
# Clean text for reuse in the workflow.
def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    cleaned = re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip()
    return cleaned or None


# Extract player ID for reuse in the workflow.
def extract_player_id(player_url: str) -> int:
    match = re.search(r"-(\d+)(?:[/?#]|$)", player_url)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(f"Could not extract a numeric player ID from {player_url!r}.")
    return int(match.group(1))


# Extract team name for reuse in the workflow.
def extract_team_name(card: Any, side: str) -> str:
    team_element = card.select_one(f".lineup__mteam.is-{side}")
    # Validate the input before continuing with later processing.
    if team_element is None:
        raise ValueError(f"Missing {side} team-name element.")
    direct_text = " ".join(
        str(node) for node in team_element.find_all(string=True, recursive=False)
    )
    team_name = clean_text(direct_text)
    # Validate the input before continuing with later processing.
    if team_name is None:
        raise ValueError(f"The {side} team name is empty.")
    return team_name


# Parse and validate starter for reuse in the workflow.
def parse_starter(player_element: Any) -> dict[str, Any]:
    link = player_element.find("a", href=True)
    position_element = player_element.select_one(".lineup__pos")
    # Validate the input before continuing with later processing.
    if link is None or position_element is None:
        raise ValueError("A selected starter is missing its player link or position.")
    full_name = clean_text(link.get("title"))
    displayed_name = clean_text(link.get_text(" ", strip=True))
    position = clean_text(position_element.get_text(" ", strip=True))
    relative_url = clean_text(link.get("href"))
    # Validate the input before continuing with later processing.
    if not all((full_name, displayed_name, position, relative_url)):
        raise ValueError("A selected starter contains an empty required field.")
    player_url = urljoin(SOURCE_URL, relative_url)
    injury_element = player_element.select_one(".lineup__inj")
    injury_status = (
        clean_text(injury_element.get_text(" ", strip=True)).upper()
        if injury_element is not None
        else None
    )
    # Validate the input before continuing with later processing.
    if injury_status not in ALLOWED_STARTER_INJURY_STATUSES:
        raise ValueError(
            f"Unexpected selected-starter injury status {injury_status!r} "
            f"for {full_name}."
        )
    return {
        "full_name": full_name,
        "displayed_name": displayed_name,
        "position": position,
        "injury_status": injury_status,
        "player_id": extract_player_id(player_url),
        "player_url": player_url,
    }


# Handle starter for reuse in the workflow.
def missing_starter() -> dict[str, Any]:
    """Represent a starter RotoWire has not yet named."""
    return {
        "full_name": "?",
        "displayed_name": "?",
        "position": "?",
        "injury_status": None,
        "player_id": None,
        "player_url": None,
    }


# Parse and validate team lineup for reuse in the workflow.
def parse_team_lineup(card: Any, side: str) -> dict[str, Any]:
    lineup_list = card.select_one(f".lineup__list.is-{side}")
    # Validate the input before continuing with later processing.
    if lineup_list is None:
        raise ValueError(f"Missing {side} lineup list.")
    status_element = lineup_list.select_one(".lineup__status")
    lineup_status = clean_text(
        status_element.get_text(" ", strip=True) if status_element else None
    )
    # Validate the input before continuing with later processing.
    if lineup_status is None:
        raise ValueError(f"The {side} lineup status is missing or empty.")
    collecting_starters = False
    injuries_heading_seen = False
    players: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for child in lineup_list.find_all("li", recursive=False):
        child_classes = set(child.get("class", []))
        if "lineup__status" in child_classes:
            collecting_starters = True
            continue
        if collecting_starters and "lineup__title" in child_classes:
            heading = clean_text(child.get_text(" ", strip=True))
            if heading and heading.casefold() == "injuries":
                injuries_heading_seen = True
                break
        if collecting_starters and "lineup__player" in child_classes:
            players.append(parse_starter(child))
    # Validate the input before continuing with later processing.
    if not injuries_heading_seen:
        raise ValueError(f"The {side} lineup has no Injuries boundary heading.")
    # Validate the input before continuing with later processing.
    if len(players) > EXPECTED_STARTERS_PER_TEAM:
        raise ValueError(
            f"The {side} lineup contains {len(players)} selected starters; "
            f"expected {EXPECTED_STARTERS_PER_TEAM}."
        )
    players.extend(
        missing_starter()
        for _ in range(EXPECTED_STARTERS_PER_TEAM - len(players))
    )
    return {
        "team_name": extract_team_name(card, side),
        "side": "home" if side == "home" else "away",
        "lineup_status": lineup_status,
        "players": players,
    }


# Parse and validate match card for reuse in the workflow.
def parse_match_card(card: Any) -> dict[str, Any]:
    match_time_element = card.select_one(".lineup__time")
    match_time = clean_text(
        match_time_element.get_text(" ", strip=True) if match_time_element else None
    )
    # Validate the input before continuing with later processing.
    if match_time is None:
        raise ValueError("A lineup card is missing its displayed match time.")
    return {
        "match_time": match_time,
        "home": parse_team_lineup(card, "home"),
        "away": parse_team_lineup(card, "visit"),
    }


# Parse and validate lineup page for reuse in the workflow.
def parse_lineup_page(html: str) -> list[dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select(LINEUP_CARD_SELECTOR)
    # Validate the input before continuing with later processing.
    if len(cards) != EXPECTED_MATCH_COUNT:
        raise ValueError(
            f"Found {len(cards)} Bundesliga lineup cards; expected "
            f"exactly {EXPECTED_MATCH_COUNT}."
        )
    return [parse_match_card(card) for card in cards]

## 4. Load the live RotoWire page

The explicit wait requires all nine cards and at least 11 player rows on each side. The browser is always closed, including after a timeout or parsing failure.

In [4]:
# Handle lineup cards ready for reuse in the workflow.
def all_lineup_cards_ready(current_driver: Any) -> bool:
    cards = current_driver.find_elements(By.CSS_SELECTOR, LINEUP_CARD_SELECTOR)
    if len(cards) != EXPECTED_MATCH_COUNT:
        return False
    # Process each available item while preserving the current workflow state.
    for card in cards:
        home_players = card.find_elements(
            By.CSS_SELECTOR, ".lineup__list.is-home li.lineup__player"
        )
        away_players = card.find_elements(
            By.CSS_SELECTOR, ".lineup__list.is-visit li.lineup__player"
        )
        if not home_players or not away_players:
            return False
    return True


scrape_started_at = datetime.now().astimezone()
driver = None
page_html = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(version_main=CHROME_MAJOR_VERSION)
    driver.set_window_size(1920, 1080)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    driver.get(SOURCE_URL)
    wait = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS)
    wait.until(
        lambda current_driver: current_driver.execute_script(
            "return document.readyState"
        )
        == "complete"
    )
    wait.until(all_lineup_cards_ready)
    page_html = driver.page_source
except TimeoutException as exc:
    card_count = (
        len(driver.find_elements(By.CSS_SELECTOR, LINEUP_CARD_SELECTOR))
        if driver is not None
        else 0
    )
    raise RuntimeError(
        f"Timed out waiting for all {EXPECTED_MATCH_COUNT} RotoWire lineup "
        f"cards; {card_count} card(s) were present."
    ) from exc
except WebDriverException as exc:
    raise RuntimeError(
        "Could not load RotoWire through undetected Chrome 150. Confirm that "
        "a compatible Chrome installation is available."
    ) from exc
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
            # uc 3.5.5 calls quit again from __del__; the explicit quit above is final.
            driver.quit = lambda: None
            print("Chrome driver closed.")
        except Exception as shutdown_error:
            print(
                "Chrome driver shutdown warning: "
                f"{type(shutdown_error).__name__}: {shutdown_error}"
            )
        finally:
            driver = None
# Validate the input before continuing with later processing.
if page_html is None:
    raise RuntimeError("RotoWire did not return a page to parse.")

Chrome driver closed.


## 5. Parse and validate the complete snapshot

No output file is opened until every match, team, and selected player has passed validation.

In [5]:
matches = parse_lineup_page(page_html)
scrape_finished_at = datetime.now().astimezone()
team_count = sum(2 for _match in matches)
player_count = sum(
    len(match[side]["players"])
    for match in matches
    for side in ("home", "away")
)
# Validate the input before continuing with later processing.
if team_count != EXPECTED_TEAM_COUNT:
    raise ValueError(f"Validated {team_count} teams; expected {EXPECTED_TEAM_COUNT}.")
# Validate the input before continuing with later processing.
if player_count != EXPECTED_PLAYER_COUNT:
    raise ValueError(
        f"Validated {player_count} selected starters; expected "
        f"{EXPECTED_PLAYER_COUNT}."
    )
output_data = {
    "metadata": {
        "source": SOURCE_NAME,
        "source_url": SOURCE_URL,
        "league_code": LEAGUE_CODE,
        "captured_at": scrape_started_at.isoformat(timespec="seconds"),
        "capture_finished_at": scrape_finished_at.isoformat(timespec="seconds"),
        "match_count": len(matches),
        "team_count": team_count,
        "player_count": player_count,
    },
    "matches": matches,
}
print(
    f"Validated {len(matches)} matches, {team_count} teams, and "
    f"{player_count} selected starters."
)

Validated 9 matches, 18 teams, and 198 selected starters.


## 6. Write the timestamped JSON snapshot

The temporary file is atomically renamed only after JSON serialization succeeds.

In [6]:
filename_timestamp = scrape_started_at.strftime("%Y%m%d_%H%M%S")
output_path = (
    output_directory
    / f"rotowire_bundesliga_lineups_{filename_timestamp}.json"
)
temporary_output_path = output_path.with_suffix(".json.tmp")
# Handle expected failures with a clear, actionable message.
try:
    # Use the resource only within this controlled scope.
    with temporary_output_path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(output_data, file, ensure_ascii=False, indent=2)
        file.write("\n")
    temporary_output_path.replace(output_path)
except OSError as exc:
    raise OSError(f"Could not write RotoWire snapshot {output_path}: {exc}") from exc
finally:
    temporary_output_path.unlink(missing_ok=True)
print(f"Saved RotoWire lineup snapshot: {output_path}")

Saved RotoWire lineup snapshot: C:\kickbase project\outputs\rotowire\predicted_lineups\rotowire_bundesliga_lineups_20260827_124203.json


## 7. Report the captured fixtures

The final report makes the match order, lineup statuses, counts, and output location easy to verify.

In [7]:
# Process each available item while preserving the current workflow state.
for match_number, match in enumerate(matches, start=1):
    home = match["home"]
    away = match["away"]
    print(
        f"{match_number:>2}. {match['match_time']} | "
        f"{home['team_name']} ({home['lineup_status']}) vs "
        f"{away['team_name']} ({away['lineup_status']})"
    )
flagged_starters = [
    {
        "team": match[side]["team_name"],
        "full_name": player["full_name"],
        "position": player["position"],
        "injury_status": player["injury_status"],
    }
    for match in matches
    for side in ("home", "away")
    for player in match[side]["players"]
    if player["injury_status"] is not None
]
print(f"\nSelected starters with injury flags: {len(flagged_starters)}")
# Process each available item while preserving the current workflow state.
for player in flagged_starters:
    print(
        f"- {player['team']}: {player['full_name']} "
        f"({player['position']}, {player['injury_status']})"
    )
print(f"\nJSON output: {output_path.resolve()}")

 1. August 28 2:30 PM ET | Bayern Munich (Predicted Lineup) vs VfB Stuttgart (Predicted Lineup)
 2. August 29 9:30 AM ET | SV 07 Elversberg (Predicted Lineup) vs Bayer Leverkusen (Predicted Lineup)
 3. August 29 9:30 AM ET | 1. FC Köln (Predicted Lineup) vs 1899 Hoffenheim (Predicted Lineup)
 4. August 29 9:30 AM ET | Union Berlin (Predicted Lineup) vs Eintracht Frankfurt (Predicted Lineup)
 5. August 29 9:30 AM ET | FSV Mainz 05 (Predicted Lineup) vs SC Paderborn (Predicted Lineup)
 6. August 29 9:30 AM ET | RB Leipzig (Predicted Lineup) vs Mönchengladbach (Predicted Lineup)
 7. August 29 12:30 PM ET | Borussia Dortmund (Predicted Lineup) vs Hamburger SV (Predicted Lineup)
 8. August 30 9:30 AM ET | SC Freiburg (Predicted Lineup) vs Werder Bremen (Predicted Lineup)
 9. August 30 11:30 AM ET | FC Augsburg (Predicted Lineup) vs FC Schalke 04 (Predicted Lineup)

Selected starters with injury flags: 5
- Eintracht Frankfurt: Jonathan Burkardt (FW, QUES)
- Borussia Dortmund: Ramy Bensebaini

In [8]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
